In [1]:
#day_1
import uuid
import enum
from abc import ABC, abstractmethod
from decimal import Decimal
from locale import currency


# 1. Исключения:
class AccountError(Exception):
    """Базовое исключение для всех ошибок счёта."""
    pass

class AccountFrozenError(AccountError):
    """Операция над замороженным счётом."""
    pass

class AccountClosedError(AccountError):
    """Операция над закрытым счётом."""
    pass

class InvalidOperationError(AccountError):
    """Некорректная операция (сумма, валюта, тип)."""
    pass

class InsufficientFundsError(AccountError):
    """Недостаточно средств на счёте."""
    pass


# 2. Статусы и валюты через Enum
class AccountStatus(enum.Enum):
    ACTIVE = "активный"
    FROZEN = "замороженный"
    CLOSED = "закрытый"

class Currency(enum.Enum):
    RUB = "RUB"
    USD = "USD"
    EUR = "EUR"
    KZT = "KZT"
    CNY = "CNY"

# 3. Абстрактный класс
class AbstractAccount(ABC):
    """Базовый контракт для любого банковского счёта."""

    def __init__(self, owner: str, account_id: str | None = None,
                 currency: Currency = Currency.RUB):
        self.owner = owner
        self.account_id = account_id or self._generate_id()
        self._balance = Decimal("0")
        self._status = AccountStatus.ACTIVE
        self.currency = currency

    # Абстрактные методы
    @abstractmethod
    def deposit(self, amount) -> None:
        ...

    @abstractmethod
    def withdraw(self, amount) -> None:
        ...

    @abstractmethod
    def get_account_info(self) -> str:
        ...

    # Общие утилиты
    @staticmethod
    def _generate_id() -> str:
        """Короткий UUID (8 символов)."""
        return uuid.uuid4().hex[:8].upper()

# 4. Конкретный счёт
class BankAccount(AbstractAccount):
    """Реализация банковского счёта с бизнес-правилами."""

    def __init__(self, owner: str, account_id: str | None = None,
                 currency: Currency = Currency.RUB,
                 initial_balance: float | Decimal = 0):
        # Валидация владельца
        if not owner or not isinstance(owner, str) or not owner.strip():
            raise InvalidOperationError("Имя владельца не может быть пустым")

        # Валидация валюты
        if not isinstance(currency, Currency):
            raise InvalidOperationError(
                f"Валюта должна быть из списка {[c.value for c in Currency]}"
            )

        super().__init__(owner.strip(), account_id, currency)

        # Начальное пополнение, если задано
        if initial_balance:
            self.deposit(initial_balance)

    # Валидация
    @staticmethod
    def _validate_amount(amount) -> Decimal:
        """Проверяем, что сумма - положительное число."""
        if isinstance(amount, bool):  # bool — подкласс int, отсекаем
            raise InvalidOperationError("Сумма должна быть числом, не bool")
        try:
            value = Decimal(str(amount))
        except Exception:
            raise InvalidOperationError(f"Некорректная сумма: {amount!r}")
        if value <= 0:
            raise InvalidOperationError("Сумма должна быть больше нуля")
        return value

    def _ensure_active(self) -> None:
        """Проверяем статус счёта перед операцией."""
        if self._status == AccountStatus.FROZEN:
            raise AccountFrozenError(
                f"Счёт {self.account_id} заморожен, операции запрещены"
            )
        if self._status == AccountStatus.CLOSED:
            raise AccountClosedError(
                f"Счёт {self.account_id} закрыт"
            )

    # Операции
    def deposit(self, amount) -> None:
        self._ensure_active()
        value = self._validate_amount(amount)
        self._balance += value

    def withdraw(self, amount) -> None:
        self._ensure_active()
        value = self._validate_amount(amount)
        if value > self._balance:
            raise InsufficientFundsError(
                f"Недостаточно средств: доступно {self._balance} {self.currency.value}, "
                f"запрошено {value}"
            )
        self._balance -= value

    # Управление статусом
    def freeze(self) -> None:
        if self._status == AccountStatus.CLOSED:
            raise AccountClosedError("Нельзя заморозить закрытый счёт")
        self._status = AccountStatus.FROZEN

    def activate(self) -> None:
        if self._status == AccountStatus.CLOSED:
            raise AccountClosedError("Нельзя активировать закрытый счёт")
        self._status = AccountStatus.ACTIVE

    def close(self) -> None:
        if self._balance != 0:
            raise InvalidOperationError(
                "Нельзя закрыть счёт с ненулевым балансом"
            )
        self._status = AccountStatus.CLOSED

    # Информация
    def get_account_info(self) -> str:
        return (
            f"Владелец: {self.owner}\n"
            f"Счёт: ...{self.account_id[-4:]}\n"
            f"Статус: {self._status.value}\n"
            f"Баланс: {self._balance} {self.currency.value}"
        )

    def __str__(self) -> str:
        return (
            f"[{self.__class__.__name__}] "
            f"{self.owner} | "
            f"...{self.account_id[-4:]} | "
            f"{self._status.value} | "
            f"{self._balance} {self.currency.value}"
        )

    def __repr__(self) -> str:
        return (f"BankAccount(owner={self.owner!r}, "
                f"account_id={self.account_id!r}, "
                f"currency={self.currency.value}, "
                f"balance={self._balance})")


# 5. Демонстрация
def demo() -> None:
    print("=" * 60)
    print("ДЕМОНСТРАЦИЯ РАБОТЫ БАНКОВСКИХ СЧЕТОВ")
    print("=" * 60)

    # 1. Активный счёт
    acc1 = BankAccount("Иван Петров", currency=Currency.RUB,
                       initial_balance=1000)
    print("\n[1] Создан активный счёт:")
    print(acc1)

    # 2. Замороженный счёт
    acc2 = BankAccount("Мария Сидорова", currency=Currency.USD,
                       initial_balance=500)
    acc2.freeze()
    print("\n[2] Создан и заморожен счёт:")
    print(acc2)

    # 3. Операции над замороженным
    print("\n[3] Попытка операций над замороженным счётом:")
    try:
        acc2.deposit(100)
    except AccountFrozenError as e:
        print(f"  ❄️  {e}")
    try:
        acc2.withdraw(50)
    except AccountFrozenError as e:
        print(f"  ❄️  {e}")

    # 4. Валидные операции
    print("\n[4] Валидные операции по активному счёту:")
    acc1.deposit(2500)
    print(f"  После пополнения: {acc1}")
    acc1.withdraw(300)
    print(f"  После снятия:     {acc1}")

    # 5. Ошибки
    print("\n[5] Проверка исключений:")
    for desc, action in [
        ("Отрицательное пополнение", lambda: acc1.deposit(-100)),
        ("Нулевое снятие",           lambda: acc1.withdraw(0)),
        ("Снятие больше баланса",    lambda: acc1.withdraw(999999)),
        ("Некорректная сумма",       lambda: acc1.deposit("abc")),
    ]:
        try:
            action()
            print(f"️ {desc}: ошибка НЕ сработала!")
        except AccountError as e:
            print(f" {desc}: {type(e).__name__} — {e}")

    # 6. Инфо и закрытие
    print("\n[6] Полная информация:")
    print(acc1.get_account_info())

    print("\n[7] Закрытие счёта:")
    acc1.withdraw(acc1._balance)
    acc1.close()
    print(acc1)
    try:
        acc1.deposit(10)
    except AccountClosedError as e:
        print(f"  🚫 {e}")


if __name__ == "__main__":
    demo()

In [2]:
# 1. Нельзя создать абстрактный класс
try:
    AbstractAccount("Тест")
except TypeError as e:
    print("OK:", e)

# 2. Нельзя создать наследника без реализации всех методов
class BadAccount(AbstractAccount):
    def deposit(self, amount): pass
    # забыли withdraw и get_account_info

try:
    BadAccount("Тест")
except TypeError as e:
    print("OK:", e)

✅ OK: Can't instantiate abstract class AbstractAccount with abstract methods deposit, get_account_info, withdraw
✅ OK: Can't instantiate abstract class BadAccount with abstract methods get_account_info, withdraw


In [ ]:
#day_2
from day1 import (
    BankAccount,
    AccountStatus,
    Currency,
    AccountError,
    InvalidOperationError,
    InsufficientFundsError,
)
from decimal import Decimal
from typing import Dict


# SAVINGS
class SavingsAccount(BankAccount):
    """Сберегательный счёт с min_balance и месячной ставкой."""

    def __init__(self, owner, account_id=None, currency=Currency.RUB,
                 initial_balance=0, min_balance=0, monthly_rate=0.01):
        super().__init__(owner, account_id, currency, initial_balance)

        if min_balance < 0:
            raise InvalidOperationError("min_balance не может быть отрицательным")
        if not (Decimal("0") <= Decimal(str(monthly_rate)) <= Decimal("1")):
            raise InvalidOperationError("Ставка должна быть в диапазоне 0...1")

        self.min_balance = Decimal(str(min_balance))
        self.monthly_rate = Decimal(str(monthly_rate))

    def withdraw(self, amount) -> None:
        value = self._validate_amount(amount)
        self._ensure_active()
        if self._balance - value < self.min_balance:
            raise InsufficientFundsError(
                f"Нельзя снять {value}: остаток станет ниже минимума {self.min_balance}"
            )
        self._balance -= value

    def apply_monthly_interest(self) -> Decimal:
        self._ensure_active()
        interest = (self._balance * self.monthly_rate).quantize(Decimal("0.01"))
        self._balance += interest
        return interest

    def get_account_info(self) -> str:
        base = super().get_account_info()
        return (
            f"{base}\n"
            f"Мин. остаток: {self.min_balance}\n"
            f"Ставка в месяц: {self.monthly_rate * 100}%"
        )

    def __str__(self) -> str:
        return f" {super().__str__()} (min={self.min_balance})"


# PREMIUM
class PremiumAccount(BankAccount):
    """Премиум-счёт с овердрафтом и комиссией."""

    def __init__(self, owner, account_id=None, currency=Currency.RUB,
                 initial_balance=0, overdraft_limit=50000, commission=0.005):
        super().__init__(owner, account_id, currency, initial_balance)

        if overdraft_limit < 0:
            raise InvalidOperationError("Овердрафт не может быть отрицательным")
        if not (Decimal("0") <= Decimal(str(commission)) < Decimal("1")):
            raise InvalidOperationError("Комиссия должна быть в диапазоне 0..1")

        self.overdraft_limit = Decimal(str(overdraft_limit))
        self.commission = Decimal(str(commission))

    def deposit(self, amount) -> None:
        value = self._validate_amount(amount)
        self._ensure_active()
        self._balance += value

    def withdraw(self, amount) -> None:
        value = self._validate_amount(amount)
        self._ensure_active()

        commission = (value * self.commission).quantize(Decimal("0.01"))
        total = value + commission

        if self._balance - total < -self.overdraft_limit:
            raise InsufficientFundsError(
                f"Превышен овердрафт: доступно "
                f"{self._balance + self.overdraft_limit}, "
                f"требуется {total} (включая комиссию {commission})"
            )
        self._balance -= total

    def get_account_info(self) -> str:
        base = super().get_account_info()
        available = self._balance + self.overdraft_limit
        return (
            f"{base}\n"
            f"Овердрафт: {self.overdraft_limit}\n"
            f"Комиссия: {self.commission * 100}%\n"
            f"Доступно к снятию: {available}"
        )

    def __str__(self) -> str:
        return f" {super().__str__()} (overdraft={self.overdraft_limit})"

# INVESTMENT
class InvestmentAccount(BankAccount):
    """Инвестиционный счёт с портфелем активов."""

    ALLOWED_ASSETS = {"stocks", "bonds", "etf"}

    def __init__(self, owner, account_id=None, currency=Currency.RUB,
                 initial_balance=0):
        super().__init__(owner, account_id, currency, initial_balance)
        self.portfolio: Dict[str, Decimal] = {}

    def _add_to_portfolio(self, asset_type: str, amount: Decimal) -> None:
        if asset_type not in self.ALLOWED_ASSETS:
            raise InvalidOperationError(
                f"Недопустимый тип актива: {asset_type!r}. "
                f"Разрешены: {sorted(self.ALLOWED_ASSETS)}"
            )
        self.portfolio[asset_type] = (
            self.portfolio.get(asset_type, Decimal("0")) + amount
        )

    def invest(self, asset_type: str, amount) -> None:
        value = self._validate_amount(amount)
        self._ensure_active()
        if value > self._balance:
            raise InsufficientFundsError(
                f"Недостаточно свободных средств: доступно {self._balance}, "
                f"запрошено {value}"
            )
        # сначала то, что может упасть
        self._add_to_portfolio(asset_type, value)
        self._balance -= value

    def withdraw(self, amount) -> None:
        # Снять можно только свободные средства (не из портфеля)
        super().withdraw(amount)

    def project_yearly_growth(self, growth_rates: Dict[str, Decimal]) -> Dict[str, Decimal]:
        result: Dict[str, Decimal] = {}
        total_growth = Decimal("0")
        for asset, amount in self.portfolio.items():
            rate = Decimal(str(growth_rates.get(asset, 0)))
            growth = (amount * rate).quantize(Decimal("0.01"))
            result[asset] = growth
            total_growth += growth
        result["TOTAL"] = total_growth
        return result

    def get_account_info(self) -> str:
        base = super().get_account_info()
        lines = [base, "Портфель:"]
        if not self.portfolio:
            lines.append("  (пусто)")
        else:
            for asset, amount in self.portfolio.items():
                lines.append(f"  {asset}: {amount}")
        return "\n".join(lines)

    def __str__(self) -> str:
        invested = sum(self.portfolio.values(), Decimal("0"))
        return f"📊 {super().__str__()} (в портфеле: {invested})"


# DEMO
def demo() -> None:
    print("=" * 60)
    print("ДЕНЬ 2. ПРОДВИНУТЫЕ СЧЕТА")
    print("=" * 60)

    # Savings
    print("\n[1] SavingsAccount:")
    savings = SavingsAccount(
        "Анна Смирнова", currency=Currency.RUB,
        initial_balance=10000, min_balance=5000, monthly_rate=0.02,
    )
    print(savings)
    savings.deposit(5000)
    print(f"  После пополнения: {savings}")
    try:
        savings.withdraw(11000)
    except InsufficientFundsError as e:
        print(f"  {e}")
    interest = savings.apply_monthly_interest()
    print(f"  Начислено процентов: {interest}")
    print(f"  Итог: {savings}")

    # Premium
    print("\n[2] PremiumAccount:")
    premium = PremiumAccount(
        "Борис Иванов", currency=Currency.USD,
        initial_balance=100, overdraft_limit=500, commission=0.01,
    )
    print(premium)
    premium.withdraw(300)
    print(f"  После снятия 300: {premium}")
    try:
        premium.withdraw(400)
    except InsufficientFundsError as e:
        print(f"  {e}")

    # Investment
    print("\n[3] InvestmentAccount:")
    invest = InvestmentAccount(
        "Виктор Орлов", currency=Currency.RUB, initial_balance=100000,
    )
    invest.invest("stocks", 40000)
    invest.invest("bonds", 30000)
    invest.invest("etf", 20000)
    print(invest)
    print(invest.get_account_info())
    try:
        invest.invest("crypto", 1000)
    except InvalidOperationError as e:
        print(f"  {e}")

    print("\n  Прогноз годового роста:")
    growth = invest.project_yearly_growth({
        "stocks": Decimal("0.10"),
        "bonds": Decimal("0.04"),
        "etf": Decimal("0.07"),
    })
    for asset, amount in growth.items():
        print(f"    {asset}: {amount}")

    # Полиморфизм
    print("\n[4] Полиморфизм:")
    accounts = [savings, premium, invest]
    for acc in accounts:
        print(f"  {acc}")
    print()
    for acc in accounts:
        try:
            acc.deposit(100)
            print(f"  {type(acc).__name__}: пополнение OK")
        except AccountError as e:
            print(f"  {type(acc).__name__}: {e}")


if __name__ == "__main__":
    demo()

In [ ]:
#day3
import uuid
import enum
from abc import ABC, abstractmethod
from decimal import Decimal
from datetime import datetime
from typing import Dict, List



class AccountError(Exception): ...
class AccountFrozenError(AccountError): ...
class AccountClosedError(AccountError): ...
class InvalidOperationError(AccountError): ...
class InsufficientFundsError(AccountError): ...


class AccountStatus(enum.Enum):
    ACTIVE = "активный"
    FROZEN = "замороженный"
    CLOSED = "закрытый"


class Currency(enum.Enum):
    RUB = "RUB"
    USD = "USD"
    EUR = "EUR"
    KZT = "KZT"
    CNY = "CNY"


class AbstractAccount(ABC):
    def __init__(self, owner, account_id=None, currency=Currency.RUB):
        self.owner = owner
        self.account_id = account_id or self._generate_id()
        self._balance = Decimal("0")
        self._status = AccountStatus.ACTIVE
        self.currency = currency

    @abstractmethod
    def deposit(self, amount) -> None: ...
    @abstractmethod
    def withdraw(self, amount) -> None: ...
    @abstractmethod
    def get_account_info(self) -> str: ...

    @staticmethod
    def _generate_id() -> str:
        return uuid.uuid4().hex[:8].upper()


class BankAccount(AbstractAccount):
    def __init__(self, owner, account_id=None, currency=Currency.RUB,
                 initial_balance=0):
        if not owner or not isinstance(owner, str) or not owner.strip():
            raise InvalidOperationError("Имя владельца не может быть пустым")
        if not isinstance(currency, Currency):
            raise InvalidOperationError("Некорректная валюта")
        super().__init__(owner.strip(), account_id, currency)
        if initial_balance:
            self.deposit(initial_balance)

    @staticmethod
    def _validate_amount(amount) -> Decimal:
        if isinstance(amount, bool):
            raise InvalidOperationError("Сумма должна быть числом, не bool")
        try:
            value = Decimal(str(amount))
        except Exception:
            raise InvalidOperationError(f"Некорректная сумма: {amount!r}")
        if value <= 0:
            raise InvalidOperationError("Сумма должна быть больше нуля")
        return value

    def _ensure_active(self) -> None:
        if self._status == AccountStatus.FROZEN:
            raise AccountFrozenError(f"Счёт {self.account_id} заморожен")
        if self._status == AccountStatus.CLOSED:
            raise AccountClosedError(f"Счёт {self.account_id} закрыт")

    def deposit(self, amount) -> None:
        self._ensure_active()
        self._balance += self._validate_amount(amount)

    def withdraw(self, amount) -> None:
        self._ensure_active()
        value = self._validate_amount(amount)
        if value > self._balance:
            raise InsufficientFundsError(
                f"Недостаточно средств: {self._balance} < {value}"
            )
        self._balance -= value

    def freeze(self) -> None:
        if self._status == AccountStatus.CLOSED:
            raise AccountClosedError("Нельзя заморозить закрытый счёт")
        self._status = AccountStatus.FROZEN

    def activate(self) -> None:
        if self._status == AccountStatus.CLOSED:
            raise AccountClosedError("Нельзя активировать закрытый счёт")
        self._status = AccountStatus.ACTIVE

    def close(self) -> None:
        if self._balance != 0:
            raise InvalidOperationError("Нельзя закрыть счёт с балансом ≠ 0")
        self._status = AccountStatus.CLOSED

    def get_account_info(self) -> str:
        return (
            f"Владелец: {self.owner}\n"
            f"Счёт: ...{self.account_id[-4:]}\n"
            f"Статус: {self._status.value}\n"
            f"Баланс: {self._balance} {self.currency.value}"
        )

    def __str__(self) -> str:
        return (f"[{self.__class__.__name__}] {self.owner} | "
                f"...{self.account_id[-4:]} | {self._status.value} | "
                f"{self._balance} {self.currency.value}")


# ПРОДВИНУТЫЕ СЧЕТА ИЗ ДНЯ 2
class SavingsAccount(BankAccount):
    def __init__(self, owner, account_id=None, currency=Currency.RUB,
                 initial_balance=0, min_balance=0, monthly_rate=0.01):
        super().__init__(owner, account_id, currency, initial_balance)
        if min_balance < 0:
            raise InvalidOperationError("min_balance < 0")
        if not (Decimal("0") <= Decimal(str(monthly_rate)) <= Decimal("1")):
            raise InvalidOperationError("Ставка вне 0..1")
        self.min_balance = Decimal(str(min_balance))
        self.monthly_rate = Decimal(str(monthly_rate))

    def withdraw(self, amount) -> None:
        value = self._validate_amount(amount)
        self._ensure_active()
        if self._balance - value < self.min_balance:
            raise InsufficientFundsError(
                f"Остаток станет ниже минимума {self.min_balance}"
            )
        self._balance -= value

    def apply_monthly_interest(self) -> Decimal:
        self._ensure_active()
        interest = (self._balance * self.monthly_rate).quantize(Decimal("0.01"))
        self._balance += interest
        return interest

    def __str__(self) -> str:
        return f" {super().__str__()} (min={self.min_balance})"


class PremiumAccount(BankAccount):
    def __init__(self, owner, account_id=None, currency=Currency.RUB,
                 initial_balance=0, overdraft_limit=50000, commission=0.005):
        super().__init__(owner, account_id, currency, initial_balance)
        if overdraft_limit < 0:
            raise InvalidOperationError("Овердрафт < 0")
        if not (Decimal("0") <= Decimal(str(commission)) < Decimal("1")):
            raise InvalidOperationError("Комиссия вне 0..1")
        self.overdraft_limit = Decimal(str(overdraft_limit))
        self.commission = Decimal(str(commission))

    def withdraw(self, amount) -> None:
        value = self._validate_amount(amount)
        self._ensure_active()
        commission = (value * self.commission).quantize(Decimal("0.01"))
        total = value + commission
        if self._balance - total < -self.overdraft_limit:
            raise InsufficientFundsError(
                f"Превышен овердрафт: доступно "
                f"{self._balance + self.overdraft_limit}, требуется {total}"
            )
        self._balance -= total

    def __str__(self) -> str:
        return f" {super().__str__()} (overdraft={self.overdraft_limit})"


class InvestmentAccount(BankAccount):
    ALLOWED_ASSETS = {"stocks", "bonds", "etf"}

    def __init__(self, owner, account_id=None, currency=Currency.RUB,
                 initial_balance=0):
        super().__init__(owner, account_id, currency, initial_balance)
        self.portfolio: Dict[str, Decimal] = {}

    def _add_to_portfolio(self, asset_type: str, amount: Decimal) -> None:
        if asset_type not in self.ALLOWED_ASSETS:
            raise InvalidOperationError(f"Актив {asset_type!r} не разрешён")
        self.portfolio[asset_type] = (
            self.portfolio.get(asset_type, Decimal("0")) + amount
        )

    def invest(self, asset_type: str, amount) -> None:
        value = self._validate_amount(amount)
        self._ensure_active()
        if value > self._balance:
            raise InsufficientFundsError(
                f"Свободно {self._balance}, запрошено {value}"
            )
        self._add_to_portfolio(asset_type, value)
        self._balance -= value

    def withdraw(self, amount) -> None:
        super().withdraw(amount)

    def project_yearly_growth(self, growth_rates) -> Dict[str, Decimal]:
        result, total = {}, Decimal("0")
        for asset, amount in self.portfolio.items():
            rate = Decimal(str(growth_rates.get(asset, 0)))
            growth = (amount * rate).quantize(Decimal("0.01"))
            result[asset] = growth
            total += growth
        result["TOTAL"] = total
        return result

    def __str__(self) -> str:
        invested = sum(self.portfolio.values(), Decimal("0"))
        return f"📊 {super().__str__()} (в портфеле: {invested})"



# ДЕНЬ 3. КЛИЕНТ
class ClientStatus(enum.Enum):
    ACTIVE = "активный"
    BLOCKED = "заблокирован"


class Client:
    def __init__(self, full_name, birth_year, contacts, pin,
                 client_id=None):
        if not full_name or not full_name.strip():
            raise InvalidOperationError("ФИО не может быть пустым")

        age = datetime.now().year - birth_year
        if age < 18:
            raise InvalidOperationError(
                f"Клиент несовершеннолетний: {age} лет"
            )
        if not isinstance(pin, str) or not pin.isdigit() or len(pin) != 4:
            raise InvalidOperationError("PIN — 4 цифры")
        if not isinstance(contacts, dict):
            raise InvalidOperationError("contacts должен быть словарём")

        self.full_name = full_name.strip()
        self.birth_year = birth_year
        self.age = age
        self.contacts = contacts
        self.pin = pin
        self.client_id = client_id or ("C" + uuid.uuid4().hex[:6].upper())
        self.status = ClientStatus.ACTIVE
        self.account_numbers: List[str] = []
        self.failed_attempts = 0
        self.suspicious_actions: List[str] = []

    def mark_suspicious(self, reason: str) -> None:
        stamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        self.suspicious_actions.append(f"[{stamp}] {reason}")

    def __str__(self) -> str:
        accs = ", ".join(self.account_numbers) or "нет"
        return (f"👤 {self.full_name} (ID {self.client_id}, {self.age} лет, "
                f"{self.status.value}), счета: {accs}")



# ДЕНЬ 3. БАНК
class Bank:
    MAX_FAILED_ATTEMPTS = 3
    NIGHT_START = 0
    NIGHT_END = 5

    ACCOUNT_TYPES = {
        "basic": BankAccount,
        "savings": SavingsAccount,
        "premium": PremiumAccount,
        "investment": InvestmentAccount,
    }

    def __init__(self, name: str):
        self.name = name
        self.clients: Dict[str, Client] = {}
        self.accounts: Dict[str, AbstractAccount] = {}
        self._current_hour: int | None = None  # для тестов

    #  утилиты
    def _get_client(self, client_id: str) -> Client:
        if client_id not in self.clients:
            raise InvalidOperationError(f"Клиент {client_id} не найден")
        return self.clients[client_id]

    def _get_account(self, account_id: str) -> AbstractAccount:
        if account_id not in self.accounts:
            raise InvalidOperationError(f"Счёт {account_id} не найден")
        return self.accounts[account_id]

    def _is_night(self) -> bool:
        hour = (self._current_hour if self._current_hour is not None
                else datetime.now().hour)
        return self.NIGHT_START <= hour < self.NIGHT_END

    def _check_night_restriction(self) -> None:
        if self._is_night():
            raise InvalidOperationError("Операции запрещены с 00:00 до 05:00")

    #  клиенты
    def add_client(self, full_name, birth_year, contacts, pin) -> Client:
        client = Client(full_name, birth_year, contacts, pin)
        self.clients[client.client_id] = client
        return client

    # счета
    def open_account(self, client_id, account_type="basic",
                     **kwargs) -> AbstractAccount:
        client = self._get_client(client_id)
        if client.status == ClientStatus.BLOCKED:
            raise InvalidOperationError("Клиент заблокирован")

        cls = self.ACCOUNT_TYPES.get(account_type)
        if cls is None:
            raise InvalidOperationError(
                f"Неизвестный тип: {account_type!r}. "
                f"Доступны: {list(self.ACCOUNT_TYPES)}"
            )

        account = cls(client.full_name, **kwargs)
        self.accounts[account.account_id] = account
        client.account_numbers.append(account.account_id)
        return account

    def close_account(self, client_id, account_id) -> None:
        client = self._get_client(client_id)
        if account_id not in client.account_numbers:
            raise InvalidOperationError("Счёт не принадлежит клиенту")
        self._get_account(account_id).close()
        client.account_numbers.remove(account_id)
        del self.accounts[account_id]

    def freeze_account(self, client_id, account_id) -> None:
        client = self._get_client(client_id)
        if account_id not in client.account_numbers:
            raise InvalidOperationError("Счёт не принадлежит клиенту")
        self._get_account(account_id).freeze()

    def unfreeze_account(self, client_id, account_id) -> None:
        client = self._get_client(client_id)
        if account_id not in client.account_numbers:
            raise InvalidOperationError("Счёт не принадлежит клиенту")
        self._get_account(account_id).activate()

    #  безопасность
    def authenticate_client(self, client_id: str, pin: str) -> bool:
        client = self._get_client(client_id)

        if client.status == ClientStatus.BLOCKED:
            raise InvalidOperationError(
                f"Клиент {client.full_name} заблокирован"
            )

        if pin == client.pin:
            client.failed_attempts = 0
            return True

        client.failed_attempts += 1
        client.mark_suspicious(
            f"Неверный PIN (попытка {client.failed_attempts})"
        )

        if client.failed_attempts >= self.MAX_FAILED_ATTEMPTS:
            client.status = ClientStatus.BLOCKED
            client.mark_suspicious(
                f"Заблокирован после {client.failed_attempts} попыток"
            )
            raise InvalidOperationError(
                f"Превышено число попыток. {client.full_name} заблокирован."
            )

        remaining = self.MAX_FAILED_ATTEMPTS - client.failed_attempts
        raise InvalidOperationError(
            f"Неверный PIN. Осталось попыток: {remaining}"
        )

    #  операции с ночным запретом
    def deposit(self, client_id, account_id, amount) -> None:
        self._check_night_restriction()
        client = self._get_client(client_id)
        if client.status == ClientStatus.BLOCKED:
            raise InvalidOperationError("Клиент заблокирован")
        if account_id not in client.account_numbers:
            raise InvalidOperationError("Счёт не принадлежит клиенту")
        self._get_account(account_id).deposit(amount)

    def withdraw(self, client_id, account_id, amount) -> None:
        self._check_night_restriction()
        client = self._get_client(client_id)
        if client.status == ClientStatus.BLOCKED:
            raise InvalidOperationError("Клиент заблокирован")
        if account_id not in client.account_numbers:
            raise InvalidOperationError("Счёт не принадлежит клиенту")
        self._get_account(account_id).withdraw(amount)

    #  поиск и аналитика
    def search_accounts(self, owner=None, currency=None,
                        status=None) -> List[AbstractAccount]:
        results = []
        for acc in self.accounts.values():
            if owner and owner.lower() not in acc.owner.lower():
                continue
            if currency and acc.currency != currency:
                continue
            if status and acc._status != status:
                continue
            results.append(acc)
        return results

    def get_total_balance(self) -> Decimal:
        return sum(
            (acc._balance for acc in self.accounts.values()),
            Decimal("0"),
        )

    def get_clients_ranking(self) -> List[Dict]:
        data = []
        for client in self.clients.values():
            total = sum(
                (self.accounts[aid]._balance
                 for aid in client.account_numbers
                 if aid in self.accounts),
                Decimal("0"),
            )
            data.append({
                "client_id": client.client_id,
                "full_name": client.full_name,
                "total_balance": total,
            })
        data.sort(key=lambda r: r["total_balance"], reverse=True)
        return data



# ДЕМОНСТРАЦИЯ ДНЯ 3
def demo() -> None:
    print("=" * 60)
    print("ДЕНЬ 3. СИСТЕМА БАНКА")
    print("=" * 60)

    bank = Bank("Учебный Банк")

    # 1. Несовершеннолетний
    print("\n[1] Несовершеннолетний клиент:")
    try:
        bank.add_client(
            "Маленький Петя",
            birth_year=datetime.now().year - 15,
            contacts={"phone": "+7-000"}, pin="1234",
        )
    except InvalidOperationError as e:
        print(f"   {e}")

    # 2. Нормальные клиенты
    print("\n[2] Регистрация клиентов:")
    ivan = bank.add_client(
        "Иван Петров", 1990,
        {"phone": "+7-900-111-22-33", "email": "ivan@mail.ru"}, "1111",
    )
    maria = bank.add_client(
        "Мария Сидорова", 1985,
        {"phone": "+7-900-444-55-66"}, "2222",
    )
    print(f"   {ivan}")
    print(f"   {maria}")

    # 3. Открытие счетов
    print("\n[3] Открытие счетов:")
    acc1 = bank.open_account(ivan.client_id, "savings",
                             currency=Currency.RUB,
                             initial_balance=50000, min_balance=10000)
    acc2 = bank.open_account(ivan.client_id, "premium",
                             currency=Currency.USD,
                             initial_balance=100, overdraft_limit=200)
    acc3 = bank.open_account(maria.client_id, "basic",
                             currency=Currency.RUB,
                             initial_balance=5000)
    print(f"   {acc1}")
    print(f"   {acc2}")
    print(f"   {acc3}")

    # 4. Три неверных попытки
    print("\n[4] Три неверных попытки входа (Иван):")
    for attempt in range(1, 4):
        try:
            bank.authenticate_client(ivan.client_id, "0000")
        except InvalidOperationError as e:
            print(f"  Попытка {attempt}: {e}")

    # 5. Состояние и лог
    print("\n[5] Состояние Ивана и подозрительные действия:")
    print(f"  {ivan}")
    for action in ivan.suspicious_actions:
        print(f"    {action}")

    # 6. Заблокированный не может оперировать
    print("\n[6] Иван заблокирован — операция не проходит:")
    try:
        bank.deposit(ivan.client_id, acc1.account_id, 1000)
    except InvalidOperationError as e:
        print(f"   {e}")

    # 7. Мария работает
    print("\n[7] Мария входит и делает операции:")
    bank.authenticate_client(maria.client_id, "2222")
    print("   Вход выполнен")
    bank.deposit(maria.client_id, acc3.account_id, 3000)
    bank.withdraw(maria.client_id, acc3.account_id, 1000)
    print(f"  {acc3}")

    # 8. Заморозка
    print("\n[8] Заморозка счёта Марии:")
    bank.freeze_account(maria.client_id, acc3.account_id)
    try:
        bank.deposit(maria.client_id, acc3.account_id, 500)
    except AccountFrozenError as e:
        print(f"    {e}")

    # 9. Разморозка
    print("\n[9] Разморозка:")
    bank.unfreeze_account(maria.client_id, acc3.account_id)
    bank.deposit(maria.client_id, acc3.account_id, 500)
    print(f"  {acc3}")

    # 10. Ночной режим
    print("\n[10] Ночное ограничение (симулируем 03:00):")
    bank._current_hour = 3
    try:
        bank.deposit(maria.client_id, acc3.account_id, 100)
    except InvalidOperationError as e:
        print(f"  {e}")
    bank._current_hour = 12
    bank.deposit(maria.client_id, acc3.account_id, 100)
    print(f"  (12:00) {acc3}")
    bank._current_hour = None

    # 11. Поиск и аналитика
    print("\n[11] Поиск и аналитика:")
    print(f"  RUB-счетов: {len(bank.search_accounts(currency=Currency.RUB))}")
    print(f"  Счетов Ивана: {len(bank.search_accounts(owner='иван'))}")
    print(f"  Общий баланс банка: {bank.get_total_balance()}")
    print("  Рейтинг клиентов:")
    for i, row in enumerate(bank.get_clients_ranking(), start=1):
        print(f"    {i}. {row['full_name']}: {row['total_balance']}")


if __name__ == "__main__":
    demo()